# Model Selection

In diesem Notebook werden verschiedene Klassifikationsmodelle für die Human Activity Recognition Data Challenge verglichen.

Die Daten wurden bereits in `notebooks/02_model_preparation.ipynb` vorbereitet. Für die Modellwahl verwenden wir `data/processed/train_data.csv`. Das finale Testset `data/processed/test_data.csv` bleibt für die spätere abschließende Bewertung reserviert.

**Grundprinzip:** Alle Modelle werden mit derselben Datenbasis, derselben Group-Cross-Validation und denselben Metriken verglichen.

## Notebook-Aufbau

1. Setup und Imports
2. Pfade und vorbereitete Daten laden
3. Features, Zielvariable und Groups vorbereiten
4. Group-Cross-Validation definieren
5. Gemeinsame Evaluationsfunktion
6. Logistic Regression als erstes Baseline-Modell
7. Weitere Modelle: RBF SVM, MLPClassifier, XGBoost usw.
8. Später: Gesamtvergleich, Testset-Auswertung und finale Prediction

## 1. Setup und Imports

Hier werden alle Bibliotheken geladen, die für Datenverarbeitung, Modelltraining und Modellbewertung benötigt werden.

In [1]:
from pathlib import Path
import time
import warnings

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

## 2. Projektpfade

Das Notebook liegt in `notebooks/model_selection`. Deshalb wird der Projektordner über zwei Parent-Ebenen bestimmt.

Erwartetes Ergebnis: `PROJECT_ROOT` zeigt auf den Ordner `DataChallenge`.

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: f:\THB\DataChallenge


### 2.1 Pfade zu den vorbereiteten Dateien

Die vorbereiteten Dateien aus `02_model_preparation.ipynb` liegen in `data/processed`:

- `train_data.csv`: Trainingsdaten für Modellwahl und Cross-Validation
- `test_data.csv`: reserviertes finales Testset
- `feature_columns.txt`: Liste der 561 echten Feature-Spalten

In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_data.csv"
TEST_PATH = PROCESSED_DIR / "test_data.csv"
FEATURE_COLUMNS_PATH = PROCESSED_DIR / "feature_columns.txt"

print(TRAIN_PATH.exists(), TRAIN_PATH)
print(TEST_PATH.exists(), TEST_PATH)
print(FEATURE_COLUMNS_PATH.exists(), FEATURE_COLUMNS_PATH)

True f:\THB\DataChallenge\data\processed\train_data.csv
True f:\THB\DataChallenge\data\processed\test_data.csv
True f:\THB\DataChallenge\data\processed\feature_columns.txt


## 3. Vorbereitete Daten laden

`train_data` wird für Cross-Validation und Modellwahl verwendet. `test_data` bleibt zunächst reserviert und wird erst nach der Modellwahl zur finalen Bewertung genutzt.

In [4]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

print("train_data:", train_data.shape)
print("test_data:", test_data.shape)

display(train_data.head())
display(test_data.head())

train_data: (5867, 563)
test_data: (1485, 563)


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1.0,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1.0,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1.0,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1.0,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1.0,STANDING


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,activity
0,0.278791,-0.032477,-0.145829,-0.993050,-0.938822,-0.928840,-0.993505,-0.935597,-0.916592,-0.937860,...,-0.939560,-0.051193,0.102632,0.066183,0.944358,-0.838642,0.209054,0.005301,27.0,STANDING
1,0.275709,-0.017983,-0.102424,-0.995569,-0.981350,-0.978256,-0.995906,-0.981642,-0.981780,-0.942066,...,-0.844675,0.099949,-0.033375,0.629909,0.409820,-0.830250,0.215456,0.014507,27.0,STANDING
2,0.277683,-0.021163,-0.107035,-0.995089,-0.982616,-0.985356,-0.996059,-0.983539,-0.985157,-0.934012,...,-0.818973,-0.108542,0.221648,0.784523,-0.281809,-0.829194,0.216168,0.013870,27.0,STANDING
3,0.277301,-0.015994,-0.098619,-0.994266,-0.978326,-0.976918,-0.995092,-0.978368,-0.975008,-0.934012,...,-0.753174,0.033209,0.391622,0.878145,-0.952204,-0.827296,0.217438,0.012735,27.0,STANDING
4,0.279601,-0.013969,-0.090564,-0.995235,-0.987974,-0.989725,-0.995816,-0.988687,-0.989033,-0.940833,...,-0.921732,-0.013611,-0.501153,0.860905,-0.182502,-0.827697,0.217023,0.009989,27.0,STANDING


### 3.1 Kurzer Strukturcheck

Hier wird geprüft, ob die wichtigen Spalten `subject` und `activity` vorhanden sind und ob die CSV-Dateien korrekt eingelesen wurden.

In [5]:
print("train columns contain subject:", "subject" in train_data.columns)
print("train columns contain activity:", "activity" in train_data.columns)

print("test columns contain subject:", "subject" in test_data.columns)
print("test columns contain activity:", "activity" in test_data.columns)

print(train_data.columns[:10])
print(train_data.columns[-10:])

train columns contain subject: True
train columns contain activity: True
test columns contain subject: True
test columns contain activity: True
Index(['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
       'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X'],
      dtype='str')
Index(['fBodyBodyGyroJerkMag-kurtosis()', 'angle(tBodyAccMean,gravity)',
       'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)', 'subject', 'activity'],
      dtype='str')


## 4. Feature-Spalten laden

`feature_columns.txt` enthält die Namen der 561 Sensor-Features. Diese Liste verhindert, dass `subject` oder `activity` versehentlich als Modellfeature verwendet werden.

In [6]:
with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as file:
    feature_cols = [line.strip() for line in file if line.strip()]

print("Anzahl Features:", len(feature_cols))
print("Erste Features:", feature_cols[:5])
print("Letzte Features:", feature_cols[-5:])

Anzahl Features: 561
Erste Features: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y']
Letzte Features: ['angle(tBodyGyroMean,gravityMean)', 'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)', 'angle(Y,gravityMean)', 'angle(Z,gravityMean)']


## 5. Features, Zielvariable und Groups vorbereiten

Für scikit-learn werden die Daten in drei Teile getrennt:

- `X`: Eingabedaten, also die 561 Sensor-Features
- `y`: Zielvariable, also die Aktivität aus `activity`
- `groups`: Personen-ID aus `subject` für die Group-Cross-Validation

Wichtig: `subject` ist **kein** Modellfeature. Es wird nur für die Gruppierung verwendet.

In [7]:
# Name der Spalte, die vorhergesagt werden soll
TARGET_COL = "activity"

# Name der Spalte, die angibt, von welcher Person die Messung stammt
GROUP_COL = "subject"


# X_train enthält nur die echten Sensor-Features.
# Das sind die Eingaben für das Modell.
X_train = train_data[feature_cols]

# y_train enthält die richtigen Aktivitätslabels.
# Das ist das, was das Modell lernen soll.
y_train = train_data[TARGET_COL]

# groups_train enthält die Personen-ID jeder Zeile.
# Das brauchen wir später für GroupKFold.
groups_train = train_data[GROUP_COL]


# X_test enthält die Sensor-Features des finalen Testsets.
X_test = test_data[feature_cols]

# y_test enthält die richtigen Labels des finalen Testsets.
y_test = test_data[TARGET_COL]

# groups_test enthält die Personen-IDs im finalen Testset.
groups_test = test_data[GROUP_COL]

### 5.1 Kontrollausgabe

Die wichtigsten Prüfungen sind:

- `X_train` hat 561 Features.
- `subject` ist nicht in `X_train` enthalten.
- `activity` ist nicht in `X_train` enthalten.
- Train- und Test-Subjects sind sauber getrennt.

In [8]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("groups_train:", groups_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("groups_test:", groups_test.shape)

print("\nIst 'subject' in X_train?", "subject" in X_train.columns)
print("Ist 'activity' in X_train?", "activity" in X_train.columns)

print("\nTrain subjects:", sorted(groups_train.unique()))
print("Test subjects:", sorted(groups_test.unique()))

print("\nActivity distribution train:")
print(y_train.value_counts())

print("\nActivity distribution test:")
print(y_test.value_counts())

X_train: (5867, 561)
y_train: (5867,)
groups_train: (5867,)
X_test: (1485, 561)
y_test: (1485,)
groups_test: (1485,)

Ist 'subject' in X_train? False
Ist 'activity' in X_train? False

Train subjects: [np.float64(1.0), np.float64(3.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(11.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(19.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(25.0), np.float64(26.0)]
Test subjects: [np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0)]

Activity distribution train:
activity
LAYING                1114
STANDING              1091
SITTING               1022
WALKING                997
WALKING_UPSTAIRS       857
WALKING_DOWNSTAIRS     786
Name: count, dtype: int64

Activity distribution test:
activity
LAYING                293
STANDING              283
SITTING               264
WALKING               229
WALKING_UPSTAIRS      216
WALKING_DOWNSTAI

## 6. Group-Cross-Validation

Für die Modellwahl wird `GroupKFold` verwendet. Dadurch bleiben alle Messfenster derselben Person immer gemeinsam in einem Fold.

Das verhindert, dass Daten derselben Person gleichzeitig in Training und Validierung landen.

In [9]:
cv = GroupKFold(n_splits=5)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

## 7. Gemeinsame Evaluationsfunktion

Diese Funktion wird für alle Modelle wiederverwendet. Sie führt Cross-Validation aus und gibt die wichtigsten Kennzahlen als Dictionary zurück:

- mittlere Accuracy
- Standardabweichung der Accuracy
- mittlerer Macro-F1
- Standardabweichung des Macro-F1
- Trainings- und Bewertungszeit

In [10]:
def evaluate_model_cv(model_name, model, X, y, groups, cv, scoring):
    start_time = time.time()
    
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        groups=groups,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )
    
    total_time = time.time() - start_time
    
    return {
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
        "fit_time_mean": scores["fit_time"].mean(),
        "score_time_mean": scores["score_time"].mean(),
        "total_time": total_time
    }

# 8. Logistic Regression

Die Logistic Regression dient als erstes lineares Baseline-Modell.

Sie ist schnell, gut interpretierbar und eignet sich als Vergleichspunkt für komplexere Modelle. Da die Aufgabe mehrere Aktivitätsklassen hat, wird Logistic Regression hier als Mehrklassen-Klassifikator verwendet.

## 8.1 Logistic Regression mit verschiedenen Regularisierungswerten

Der wichtigste getestete Hyperparameter ist `C`.

- Kleines `C`: stärkere Regularisierung
- Großes `C`: schwächere Regularisierung

Hier werden mehrere Werte getestet, um eine erste solide Baseline zu bekommen.

In [11]:
models = {
    "LogReg C=0.01": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=0.01,
            random_state=42
        ))
    ]),
    
    "LogReg C=0.1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=0.1,
            random_state=42
        ))
    ]),
    
    "LogReg C=1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=1.0,
            random_state=42
        ))
    ]),
    
    "LogReg C=10": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=10.0,
            random_state=42
        ))
    ]),
    
    "LogReg C=100": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=100.0,
            random_state=42
        ))
    ]),
}

## 8.2 Cross-Validation für Logistic Regression

Die folgenden Modelle werden mit derselben Group-Cross-Validation ausgewertet. Die Ergebnistabelle wird als `log_reg_results_df` gespeichert, damit sie später mit Ergebnissen anderer Modelltypen kombiniert werden kann.

In [12]:
results = []

for model_name, model in models.items():
    print(f"Evaluating: {model_name}")
    
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X=X_train,
        y=y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring
    )
    
    results.append(result)

log_reg_results_df = pd.DataFrame(results)
log_reg_results_df = log_reg_results_df.sort_values(by="accuracy_mean", ascending=False)

display(log_reg_results_df)

Evaluating: LogReg C=0.01
Evaluating: LogReg C=0.1
Evaluating: LogReg C=1
Evaluating: LogReg C=10
Evaluating: LogReg C=100


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,fit_time_mean,score_time_mean,total_time
1,LogReg C=0.1,0.915994,0.058786,0.913188,0.055669,1.089569,0.021593,3.827959
2,LogReg C=1,0.915274,0.060308,0.912913,0.057117,1.595724,0.021442,2.394355
0,LogReg C=0.01,0.914818,0.050452,0.910697,0.048949,2.042664,0.028550,7.082343
3,LogReg C=10,0.902893,0.059197,0.900707,0.056382,1.607081,0.023385,2.290379
4,LogReg C=100,0.900815,0.054662,0.897746,0.051047,1.163244,0.028311,1.897353


## 8.3 Ergebnis Logistic Regression

Die Logistic Regression wurde mit verschiedenen Regularisierungsstärken getestet. Nach dem bisherigen Lauf war `C=0.1` die beste Variante in der Group-Cross-Validation.

Der Unterschied zu `C=1.0` war sehr klein. Deshalb ist Logistic Regression zunächst als solides Baseline-Modell abgeschlossen. Feineres Tuning kann später erfolgen, wenn mehrere Modelltypen verglichen wurden.

### 8.4 Bestes Logistic-Regression-Modell speichern

Hier kann später eine kurze Code-Zelle eingefügt werden, die die beste Logistic-Regression-Variante als `best_log_reg_model` speichert.

In [13]:
best_log_reg_model = models["LogReg C=0.1"]
best_log_reg_name = "LogReg C=0.1"

print("Best Logistic Regression model:", best_log_reg_name)

Best Logistic Regression model: LogReg C=0.1


# 9. RBF SVM

Als nächstes kann eine Support Vector Machine mit RBF-Kernel ergänzt werden.

Im Gegensatz zur Logistic Regression ist dieses Modell nichtlinear. Es kann komplexere Entscheidungsgrenzen lernen, ist aber meist rechenintensiver und stärker von den Hyperparametern `C` und `gamma` abhängig.

**Code wird später ergänzt.**

### 9.1 Imports

In [14]:
from sklearn.svm import SVC

### 9.2 RBF-SVM-Modelle definieren

Für die RBF SVM werden zunächst mehrere Werte für `C` getestet.  
Der Parameter `gamma` bleibt vorerst auf dem Standardwert `"scale"`.

`C` steuert die Regularisierung. Kleine Werte erzeugen glattere Entscheidungsgrenzen, größere Werte passen sich stärker an die Trainingsdaten an.

In [15]:
svm_models = {
    "RBF SVM C=0.1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=0.1,
            gamma="scale"
        ))
    ]),
    
    "RBF SVM C=1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        ))
    ]),
    
    "RBF SVM C=10": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=10.0,
            gamma="scale"
        ))
    ]),
}

### 9.3 RBF SVM per Cross-Validation auswerten

Die RBF-SVM-Modelle werden mit derselben Group-Cross-Validation bewertet wie zuvor die Logistic Regression.  
Dadurch bleiben die Ergebnisse vergleichbar.

In [16]:
svm_results = []

for model_name, model in svm_models.items():
    print(f"Evaluating: {model_name}")
    
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X=X_train,
        y=y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring
    )
    
    svm_results.append(result)

svm_results_df = pd.DataFrame(svm_results)
svm_results_df = svm_results_df.sort_values(by="accuracy_mean", ascending=False)

display(svm_results_df)

Evaluating: RBF SVM C=0.1
Evaluating: RBF SVM C=1
Evaluating: RBF SVM C=10


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,fit_time_mean,score_time_mean,total_time
2,RBF SVM C=10,0.910096,0.042039,0.905190,0.042699,3.232598,1.297134,5.200538
1,RBF SVM C=1,0.904375,0.041536,0.898817,0.043063,3.767310,1.731240,6.089389
0,RBF SVM C=0.1,0.865754,0.042913,0.858635,0.046232,7.610952,3.022622,11.348449


### 9.4 Bestes RBF-SVM-Modell speichern

Die beste getestete RBF-SVM-Variante war `C=10` mit `gamma="scale"`.  
Dieses Modell wird gespeichert, damit es später mit den besten Modellen der anderen Modelltypen verglichen werden kann.

In [17]:
best_svm_model = svm_models["RBF SVM C=10"]
best_svm_name = "RBF SVM C=10"

print("Best RBF SVM model:", best_svm_name)

Best RBF SVM model: RBF SVM C=10


# 10. MLPClassifier

Danach kann ein neuronales Netz mit `MLPClassifier` ergänzt werden.

Dieses Modell kann nichtlineare Zusammenhänge lernen, benötigt aber sorgfältigere Einstellung der Hyperparameter, z. B. Hidden-Layer-Größe, Regularisierung und maximale Iterationen.

**Code wird später ergänzt.**

### 10.1 imports

In [18]:
from sklearn.neural_network import MLPClassifier

### 10.2 MLPClassifier-Modelle definieren

Der MLPClassifier ist ein einfaches neuronales Netz aus scikit-learn.  
Für den ersten Vergleich werden kleine Netzstrukturen getestet, damit die Trainingszeit überschaubar bleibt.

Die Features werden wie bei Logistic Regression und RBF SVM vorher skaliert.

In [19]:
mlp_models = {
    "MLP hidden=(50,)": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(50,),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            max_iter=1000,
            random_state=42
        ))
    ]),
    
    "MLP hidden=(100,)": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            max_iter=1000,
            random_state=42
        ))
    ]),
    
    "MLP hidden=(100, 50)": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(100, 50),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            max_iter=1000,
            random_state=42
        ))
    ]),
}

### 10.3 MLPClassifier per Cross-Validation auswerten

Die MLPClassifier-Modelle werden mit derselben Group-Cross-Validation bewertet wie Logistic Regression und RBF SVM.

Dadurch bleiben die Ergebnisse vergleichbar. Da neuronale Netze rechenintensiver sein können, werden zunächst nur kleine Netzstrukturen getestet.

In [20]:
mlp_results = []

for model_name, model in mlp_models.items():
    print(f"Evaluating: {model_name}")
    
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X=X_train,
        y=y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring
    )
    
    mlp_results.append(result)

mlp_results_df = pd.DataFrame(mlp_results)
mlp_results_df = mlp_results_df.sort_values(by="accuracy_mean", ascending=False)

display(mlp_results_df)

Evaluating: MLP hidden=(50,)
Evaluating: MLP hidden=(100,)
Evaluating: MLP hidden=(100, 50)


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,fit_time_mean,score_time_mean,total_time
0,"MLP hidden=(50,)",0.915257,0.054363,0.912914,0.050570,11.124229,0.024670,12.121026
2,"MLP hidden=(100, 50)",0.913561,0.057606,0.911361,0.053695,9.840614,0.026204,11.540184
1,"MLP hidden=(100,)",0.913062,0.059693,0.911392,0.056278,12.819569,0.030466,14.153351


### 10.4 Bestes MLPClassifier-Modell speichern

Die beste getestete MLP-Variante war ein Netzwerk mit einem Hidden Layer und 50 Neuronen.  
Dieses Modell wird gespeichert, damit es später mit den besten Modellen der anderen Modelltypen verglichen werden kann.

In [21]:
best_mlp_model = mlp_models["MLP hidden=(50,)"]
best_mlp_name = "MLP hidden=(50,)"

print("Best MLP model:", best_mlp_name)

Best MLP model: MLP hidden=(50,)


# 11. Gesamtvergleich aller Modelle

Hier sollen später die Ergebnis-DataFrames der einzelnen Modelltypen zusammengeführt werden, z. B.:

- Logistic Regression
- RBF SVM
- MLPClassifier
- XGBoost
- Random Forest

Ziel ist eine gemeinsame Tabelle mit Accuracy, Macro-F1, Standardabweichungen und Laufzeiten.

**Code wird später ergänzt.**

In [22]:
all_results_display = all_results_df.copy()

score_cols = [
    "accuracy_mean",
    "accuracy_std",
    "f1_macro_mean",
    "f1_macro_std"
]

time_cols = [
    "fit_time_mean",
    "score_time_mean",
    "total_time"
]

for col in score_cols:
    all_results_display[col] = all_results_display[col].round(4)

for col in time_cols:
    all_results_display[col] = all_results_display[col].round(2)

display(all_results_display)

NameError: name 'all_results_df' is not defined

# 12. Finale Testset-Auswertung

Erst nachdem die Modellwahl abgeschlossen ist, wird das beste Modell auf `test_data.csv` bewertet.

Das Testset soll nicht für die laufende Modellwahl verwendet werden, sondern nur zur abschließenden Performance-Schätzung.

**Code wird später ergänzt.**

# 13. Finale Prediction für `to_predict.csv`

Zum Schluss wird das finale Modell auf allen gelabelten Daten trainiert und auf `to_predict.csv` angewendet.

Die Datei `my_prediction.csv` muss die vorhergesagten Klassen in der Originalreihenfolge enthalten.

**Code wird später ergänzt.**